# 03 PSM匹配分析

## 学习目标

- 掌握倾向得分匹配（PSM）的完整流程
- 理解共同支撑域和平衡检验
- 估计处理效应（ATT）

## 背景

通过匹配处理组和对照组中倾向得分相近的用户，我们可以构造一个伪随机实验。

---

## 步骤 1：估计倾向得分

使用Logistic回归估计每个用户接受处理的概率。

In [ ]:
# TODO: 估计倾向得分
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

df = pd.read_csv('../data/raw/ecommerce_618.csv')

# 选择协变量
covariates = ['historical_gmv', 'activity_score', 'age', 'membership_level', 'city_tier']
X = df[covariates]
y = df['treatment']

# 估计倾向得分
ps_model = LogisticRegression(max_iter=1000).fit(X, y)
df['propensity_score'] = ps_model.predict_proba(X)[:, 1]

print('倾向得分分布:')
print(df['propensity_score'].describe())

## 步骤 2：匹配

使用最近邻匹配，为每个处理组用户找到一个倾向得分最接近的对照组用户。

In [ ]:
# TODO: 执行1:1最近邻匹配
from sklearn.neighbors import NearestNeighbors

treated = df[df['treatment'] == 1].copy()
control = df[df['treatment'] == 0].copy()

# 为对照组建立NN模型
nn = NearestNeighbors(n_neighbors=1)
nn.fit(control['propensity_score'].values.reshape(-1, 1))

# 为每个处理组用户找到最近的对照组用户
distances, indices = nn.kneighbors(treated['propensity_score'].values.reshape(-1, 1))

matched_control = control.iloc[indices.flatten()].copy()
matched_control.index = treated.index

print(f'匹配前处理组: {len(treated)}, 对照组: {len(control)}')
print(f'匹配后处理组: {len(treated)}, 对照组: {len(matched_control)}')

## 步骤 3：平衡检验

检查匹配后处理组和对照组的协变量是否平衡。

In [ ]:
# TODO: 计算匹配前后的SMD
def calc_smd(df_treat, df_control, var):
    mean_t = df_treat[var].mean()
    mean_c = df_control[var].mean()
    var_t = df_treat[var].var()
    var_c = df_control[var].var()
    pooled_std = np.sqrt((var_t + var_c) / 2)
    return (mean_t - mean_c) / pooled_std

print('匹配前后SMD对比:')
print(f'{'变量':<20} {'匹配前':<10} {'匹配后':<10}')
print('-' * 40)
for var in covariates:
    smd_before = calc_smd(treated, control, var)
    smd_after = calc_smd(treated, matched_control, var)
    print(f'{var:<20} {smd_before:<10.3f} {smd_after:<10.3f}')

## 步骤 4：估计ATT

计算匹配后处理组和对照组的均值差异。

In [ ]:
# TODO: 估计ATT
att = treated['post_gmv'].mean() - matched_control['post_gmv'].mean()
print(f'PSM估计ATT: {att:.2f}')

# 与Naive估计对比
naive = df[df['treatment']==1]['post_gmv'].mean() - df[df['treatment']==0]['post_gmv'].mean()
print(f'Naive估计: {naive:.2f}')
print(f'选择偏差: {naive - att:.2f}')